# Contact-Based SIP Demo

Tiny synthetic example for the contact-based SIP MVP. The data are stop records, so the workflow starts after stop detection / visit attribution.

In [ ]:
import pandas as pd

from nomad.contact_estimation import estimate_contacts, compute_contact_weights
from nomad.metrics.metrics import social_interaction_potential

Create a few stops with user IDs, time intervals, a place ID, projected coordinates, and a simple group label.

In [ ]:
stops = pd.DataFrame(
    {
        "user_id": ["alice", "bob", "cara", "drew", "alice", "bob"],
        "start_datetime": pd.to_datetime(
            [
                "2024-01-01 09:00",
                "2024-01-01 09:30",
                "2024-01-01 09:30",
                "2024-01-01 09:45",
                "2024-01-01 11:00",
                "2024-01-01 11:15",
            ],
            utc=True,
        ),
        "end_datetime": pd.to_datetime(
            [
                "2024-01-01 10:00",
                "2024-01-01 10:15",
                "2024-01-01 10:00",
                "2024-01-01 10:05",
                "2024-01-01 12:00",
                "2024-01-01 11:45",
            ],
            utc=True,
        ),
        "location_id": ["cafe", "cafe", "library", "cafe", "park", "park"],
        "x": [0, 3, 12, 20, 40, 43],
        "y": [0, 4, 4, 20, 0, 4],
        "income_group": ["low", "low", "high", "high", "low", "low"],
    }
)

stops

Exact-location mode treats a shared `location_id` plus overlapping time as a contact. This is the mode to use when places have already been assigned, such as POIs, clusters, geohashes, or H3 cells.

In [ ]:
exact_contacts = estimate_contacts(stops)
exact_contacts

Radius-distance mode uses exact coordinate distance as the final contact condition. Here the threshold is 10 coordinate units.

In [ ]:
radius_contacts = estimate_contacts(stops, distance_threshold=10)
radius_contacts

Distance-weighted contacts use overlap duration and a linear distance decay. `compute_contact_weights` returns a named weight Series, so callers can assign or join it back onto the contact table when they want a richer table for display or downstream analysis.

In [ ]:
contact_weights = compute_contact_weights(
    radius_contacts,
    method="linear_distance",
    distance_threshold=10,
)

weighted_contacts = radius_contacts.assign(contact_weight=contact_weights)
weighted_contacts

Optional: SIP itself is now just user-level aggregation. If we want demographic or group labels for inspection or later analysis, we join them outside SIP using the user columns on each side of the contact table.

In [ ]:
user_groups = stops[["user_id", "income_group"]].drop_duplicates("user_id")

contacts_with_groups = (
    weighted_contacts
    .merge(
        user_groups.rename(columns={"user_id": "user_id_1", "income_group": "income_group_1"}),
        on="user_id_1",
    )
    .merge(
        user_groups.rename(columns={"user_id": "user_id_2", "income_group": "income_group_2"}),
        on="user_id_2",
    )
)

contacts_with_groups

Aggregate contact weights into user-level SIP. Each undirected contact is credited to both users.

In [ ]:
sip = social_interaction_potential(
    weighted_contacts,
    weight_col="contact_weight",
)

sip